## 🧪 Section 1 — Interactive Query Testing
Run individual queries and inspect retrieved context + model answer side by side.

In [ ]:
def test_query(question: str, verbose: bool = True) -> dict:
    """Run a single RAG query and return a structured result dict."""
    docs = retriever.invoke(question)

    context_blocks = []
    for doc in docs:
        block = (
            f"[Source: {doc.metadata.get('source_name', 'Unknown')} | "
            f"Section: {doc.metadata.get('section', 'Unknown')} | "
            f"Date: {doc.metadata.get('publication_date', 'Unknown')}]\n"
            f"{doc.page_content}"
        )
        context_blocks.append(block)
    context = "\n\n".join(context_blocks)

    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are a helpful housing market assistant. Answer only with information supported by the source.\n<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"Context:\n{context}\n\nQuestion: {question}\n<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )

    output = pipe(prompt, max_new_tokens=256, do_sample=False,
                  pad_token_id=tokenizer.eos_token_id)
    answer = output[0]["generated_text"].split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

    result = {
        "question": question,
        "answer": answer,
        "retrieved_docs": [
            {
                "id": d.metadata.get("id", ""),
                "section": d.metadata.get("section", ""),
                "snippet": d.page_content[:200],
            }
            for d in docs
        ],
    }

    if verbose:
        print(f"{'='*70}")
        print(f"Q: {question}")
        print(f"{'─'*70}")
        print(f"ANSWER:\n{answer}")
        print(f"{'─'*70}")
        print(f"RETRIEVED DOCS ({len(docs)}):")
        for i, d in enumerate(docs, 1):
            print(f"  [{i}] {d.metadata.get('id','')} — {d.metadata.get('section','')[:60]}")
            print(f"       {d.page_content[:120]}...")
        print(f"{'='*70}\n")

    return result


In [ ]:
# ── Predefined test battery ───────────────────────────────────────────────────
TEST_QUESTIONS = [
    # Factual recall (answers exist verbatim in the data)
    "What was the Zillow Home Value Index for Iowa City as of February 28, 2026?",
    "What is the average rent in Iowa City?",
    "Which Iowa City neighborhood had the highest home value in February 2026?",
    # Inference / multi-fact
    "How much cheaper is Iowa City rent compared to the national average in dollar terms?",
    "Were most homes in Iowa City selling above or below list price?",
    # Out-of-scope (should say it doesn't know)
    "What is the median home price in Des Moines, Iowa?",
]

results = [test_query(q) for q in TEST_QUESTIONS]


## 🔍 Section 2 — RAG Grounding Verification
These checks confirm the model is actually using retrieved context, not hallucinating.

In [ ]:
import re

def check_grounding(result: dict) -> dict:
    """
    Lightweight grounding check:
    - keyword_overlap: fraction of key answer tokens found in the retrieved docs
    - answer_cites_context: heuristic — does the answer contain facts from context?
    - retrieval_relevance: do any retrieved doc snippets share tokens with the question?
    """
    answer_tokens = set(re.findall(r"\b\w+\b", result["answer"].lower()))
    question_tokens = set(re.findall(r"\b\w+\b", result["question"].lower()))
    context_text = " ".join(d["snippet"] for d in result["retrieved_docs"]).lower()
    context_tokens = set(re.findall(r"\b\w+\b", context_text))

    # Remove stopwords for a cleaner overlap signal
    STOPWORDS = {"the","a","an","is","was","in","of","to","and","for","it","that","by",
                 "on","at","be","are","as","with","from","or","this","were","had","has"}
    answer_content = answer_tokens - STOPWORDS
    context_content = context_tokens - STOPWORDS
    question_content = question_tokens - STOPWORDS

    kw_overlap = (
        len(answer_content & context_content) / max(len(answer_content), 1)
    )
    retrieval_relevance = (
        len(question_content & context_content) / max(len(question_content), 1)
    )

    return {
        "question": result["question"][:60] + "...",
        "keyword_overlap": round(kw_overlap, 3),       # higher = more grounded
        "retrieval_relevance": round(retrieval_relevance, 3),  # higher = retriever found relevant docs
        "num_docs_retrieved": len(result["retrieved_docs"]),
        "answer_length_tokens": len(answer_tokens),
    }

grounding_scores = [check_grounding(r) for r in results]

# Pretty print
print(f"{'Question':<45} {'KW Overlap':>11} {'Ret. Rel.':>10} {'Docs':>5} {'Ans Len':>8}")
print("─" * 83)
for g in grounding_scores:
    print(f"{g['question']:<45} {g['keyword_overlap']:>11.3f} {g['retrieval_relevance']:>10.3f} "
          f"{g['num_docs_retrieved']:>5} {g['answer_length_tokens']:>8}")


In [ ]:
# ── Ablation: RAG vs. No-RAG ──────────────────────────────────────────────────
# Send the same question WITH and WITHOUT context to see if retrieval matters.

def query_no_rag(question: str) -> str:
    """Same model, no retrieved context — baseline to confirm RAG adds value."""
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are a helpful housing market assistant.\n<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{question}\n<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    out = pipe(prompt, max_new_tokens=150, do_sample=False,
               pad_token_id=tokenizer.eos_token_id)
    return out[0]["generated_text"].split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

ablation_q = "What was the Zillow Home Value Index for Iowa City as of February 28, 2026?"
rag_ans = results[0]["answer"]          # already computed above
no_rag_ans = query_no_rag(ablation_q)

print("ABLATION — same question, with vs. without RAG context")
print("="*70)
print(f"Question: {ablation_q}\n")
print(f"[WITH RAG]\n{rag_ans}\n")
print(f"[WITHOUT RAG (model memory only)]\n{no_rag_ans}")
print("\nExpect: WITH RAG should cite the exact figure ($292,103). "
      "WITHOUT RAG may hallucinate or say it doesn't know.")


## 📊 Section 3 — Retriever Metrics
Precision@K and source coverage — checks the vector store is finding the right chunks.

In [ ]:
# Ground-truth source IDs for each test question (manually labelled)
GROUND_TRUTH = {
    TEST_QUESTIONS[0]: {"housing_zillow_001"},   # ZHVI citywide
    TEST_QUESTIONS[1]: {"housing_zillow_003"},   # rent data
    TEST_QUESTIONS[2]: {"housing_zillow_002"},   # neighborhood breakdown
    TEST_QUESTIONS[3]: {"housing_zillow_003"},   # rent vs national
    TEST_QUESTIONS[4]: {"housing_zillow_001"},   # sale-to-list
    TEST_QUESTIONS[5]: set(),                    # out-of-scope — no correct source
}

def precision_at_k(result: dict, ground_truth_ids: set) -> float:
    """Fraction of retrieved docs whose source_id matches the gold set."""
    if not ground_truth_ids:
        return float("nan")  # out-of-scope: not applicable
    retrieved_ids = {d["id"] for d in result["retrieved_docs"]}
    hits = retrieved_ids & ground_truth_ids
    return len(hits) / max(len(result["retrieved_docs"]), 1)

def recall_at_k(result: dict, ground_truth_ids: set) -> float:
    """Fraction of gold sources that appear in the retrieved docs."""
    if not ground_truth_ids:
        return float("nan")
    retrieved_ids = {d["id"] for d in result["retrieved_docs"]}
    hits = retrieved_ids & ground_truth_ids
    return len(hits) / max(len(ground_truth_ids), 1)

print(f"{'Question':<48} {'P@K':>6} {'R@K':>6} {'Retrieved IDs'}")
print("─" * 85)
for r in results:
    gt = GROUND_TRUTH.get(r["question"], set())
    p = precision_at_k(r, gt)
    rec = recall_at_k(r, gt)
    ret_ids = {d["id"] for d in r["retrieved_docs"]}
    p_str = f"{p:.2f}" if p == p else " N/A"
    r_str = f"{rec:.2f}" if rec == rec else " N/A"
    print(f"{r['question'][:48]:<48} {p_str:>6} {r_str:>6}  {ret_ids}")

avg_p = [precision_at_k(r, GROUND_TRUTH.get(r["question"], set()))
         for r in results if GROUND_TRUTH.get(r["question"], set())]
avg_r = [recall_at_k(r, GROUND_TRUTH.get(r["question"], set()))
         for r in results if GROUND_TRUTH.get(r["question"], set())]
print(f"\nMean P@{TOP_K} (in-scope only): {sum(avg_p)/len(avg_p):.3f}")
print(f"Mean R@{TOP_K} (in-scope only): {sum(avg_r)/len(avg_r):.3f}")


## 📐 Section 4 — Generation Quality Metrics
ROUGE-L for lexical overlap with reference answers; answer length stats; refusal detection.

In [ ]:
# pip install rouge-score  (if not already installed)
try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("rouge-score not installed — skipping ROUGE. Run: pip install rouge-score")

# Reference answers aligned to TEST_QUESTIONS (human-written gold standard)
REFERENCE_ANSWERS = [
    "The Zillow Home Value Index for Iowa City was $292,103 as of February 28, 2026.",
    "The average rent in Iowa City was $1,308 per month as of February 28, 2026.",
    "South Pointe had the highest Zillow Home Value Index at $314,570.",
    "Iowa City average rent was $587 lower per month than the national average.",
    "Most home sales were closing below list price — 68.1 percent were under list price.",
    None,  # out-of-scope — no reference
]

if HAS_ROUGE:
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    print(f"{'Question':<48} {'ROUGE-L F1':>11} {'Ans Tokens':>11}")
    print("─" * 74)
    rouge_scores = []
    for r, ref in zip(results, REFERENCE_ANSWERS):
        if ref is None:
            print(f"{r['question'][:48]:<48} {'N/A (OOS)':>11} {len(r['answer'].split()):>11}")
            continue
        score = scorer.score(ref, r["answer"])
        rl = score["rougeL"].fmeasure
        rouge_scores.append(rl)
        tokens = len(r["answer"].split())
        print(f"{r['question'][:48]:<48} {rl:>11.3f} {tokens:>11}")
    if rouge_scores:
        print(f"\nMean ROUGE-L (in-scope): {sum(rouge_scores)/len(rouge_scores):.3f}")


In [ ]:
# ── Refusal / Hallucination Detection ─────────────────────────────────────────
# Check whether out-of-scope questions trigger an appropriate refusal
# and whether in-scope answers avoid "I don't know" hedges.

REFUSAL_PATTERNS = [
    r"i (don't|do not|don't) (have|know|find)",
    r"not (in|within) (the|my) (context|source|data|information)",
    r"(no information|not available|cannot find|unable to find)",
    r"(sorry|apolog)",
]

def classify_response(answer: str, is_out_of_scope: bool) -> str:
    lowered = answer.lower()
    is_refusal = any(re.search(p, lowered) for p in REFUSAL_PATTERNS)
    if is_out_of_scope:
        return "✅ Correct refusal" if is_refusal else "⚠️  Should have refused"
    else:
        return "⚠️  Unexpected refusal" if is_refusal else "✅ Answered"

print("Response classification:")
print(f"{'Question':<50} {'OOS?':>5}  {'Status'}")
print("─" * 78)
for r, ref in zip(results, REFERENCE_ANSWERS):
    oos = ref is None
    status = classify_response(r["answer"], oos)
    print(f"{r['question'][:50]:<50} {str(oos):>5}  {status}")


## 📋 Section 5 — Summary Dashboard
Consolidated view of all metrics across test questions.

In [ ]:
import pandas as pd

rows = []
for i, (r, ref) in enumerate(zip(results, REFERENCE_ANSWERS)):
    oos = ref is None
    gt = GROUND_TRUTH.get(r["question"], set())
    g = check_grounding(r)
    p = precision_at_k(r, gt)
    rec = recall_at_k(r, gt)
    rl_score = None
    if HAS_ROUGE and not oos:
        rl_score = scorer.score(ref, r["answer"])["rougeL"].fmeasure
    refusal_status = classify_response(r["answer"], oos)

    rows.append({
        "Q#": i + 1,
        "Question (truncated)": r["question"][:50],
        "OOS": oos,
        "KW Overlap": g["keyword_overlap"],
        "Ret. Relevance": g["retrieval_relevance"],
        "P@K": round(p, 2) if p == p else "N/A",
        "R@K": round(rec, 2) if rec == rec else "N/A",
        "ROUGE-L": round(rl_score, 3) if rl_score is not None else "N/A",
        "Status": refusal_status,
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 55)
pd.set_option("display.width", 120)
print(df.to_string(index=False))


In [ ]:
# ── Optional: visualise with matplotlib ───────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

in_scope = [r for r, ref in zip(rows, REFERENCE_ANSWERS) if ref is not None]
labels = [f"Q{r['Q#']}" for r in in_scope]
kw = [r["KW Overlap"] for r in in_scope]
rr = [r["Ret. Relevance"] for r in in_scope]
pk = [r["P@K"] if isinstance(r["P@K"], float) else 0 for r in in_scope]
rl = [r["ROUGE-L"] if isinstance(r["ROUGE-L"], float) else 0 for r in in_scope]

x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 1.5*width, kw, width, label="KW Overlap")
ax.bar(x - 0.5*width, rr, width, label="Ret. Relevance")
ax.bar(x + 0.5*width, pk, width, label="P@K")
ax.bar(x + 1.5*width, rl, width, label="ROUGE-L")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("RAG Evaluation Metrics per Test Question (in-scope only)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("rag_eval_metrics.png", dpi=150)
plt.show()
print("Chart saved to rag_eval_metrics.png")
